In [ ]:
# ── BFCL paper experiments: config ──
# Reads the runs written by scripts/bfcl_capacity_sweep.sh and prints the
# median FTR and E2E table.
SYSNAME = 'Sutradhara'
SYSNAME_SHORT = 'SD'

import os, glob, csv, re
import numpy as np

def _find_repo_root():
    env = os.environ.get('SUTRADHARA_REPO')
    if env:
        return os.path.abspath(env)
    p = os.path.abspath(os.getcwd())
    while True:
        if os.path.isfile(os.path.join(p, 'pyproject.toml')):
            return p
        parent = os.path.dirname(p)
        if parent == p:
            return os.path.abspath(os.getcwd())
        p = parent

REPO_ROOT = _find_repo_root()
BASE = os.environ.get('SUTRADHARA_EXPERIMENTS', os.path.join(REPO_ROOT, 'experiments'))
PLOT_DIR = os.path.join(REPO_ROOT, 'analysis', 'paper_plots')
os.makedirs(PLOT_DIR, exist_ok=True)

# ── BFCL run configuration (must match scripts/bfcl_capacity_sweep.sh) ──
TRACE    = os.environ.get(
    'BFCL_TRACE_NAME',
    'bfcl_trace')
MEM      = '065'
SEED     = 42
# QPS points are discovered from disk, so adding sweep points needs no edit here.
def _discover_qps():
    pat = os.path.join(BASE, TRACE, f'chunk256_qps*_mem{MEM}')
    qs = []
    for d in glob.glob(pat):
        m = re.search(r'chunk256_qps([0-9.]+)_mem', d)
        if m:
            qs.append(float(m.group(1)))
    return sorted(set(qs))

# Cap the sweep: past ~4 QPS the 56-request trace arrives in under 10s, so a
# run measures a burst rather than a load level.
QPS_MAX = float(os.environ.get('BFCL_QPS_MAX', 4.0))
QPS_LIST = [q for q in _discover_qps() if q <= QPS_MAX]
PICK_QPS = 1.5          # ablation / breakdown / KV analysis load point
TAG_BL, TAG_SD = 'baseline', 'ds_kv'
TAG_LRU, TAG_PRI = 'et', 'ds_kv_et'

print(f'REPO_ROOT  {REPO_ROOT}')
print(f'BASE       {BASE}')
print(f'TRACE      {TRACE}')
print(f'QPS_LIST   {QPS_LIST}')


def run_dir(qps, tag, seed=SEED, trace=None, required=True):
    """Latest run dir for (qps, tag). Layout written by run_experiments.py:
    experiments/{trace}/chunk256_qps{qps}_mem{MEM}/{tag}/seed{seed}/{ts}/"""
    parent = os.path.join(BASE, trace or TRACE, f'chunk256_qps{qps}_mem{MEM}', tag)
    if seed is not None:
        parent = os.path.join(parent, f'seed{seed}')
    runs = sorted(d for d in glob.glob(parent + '/*')
                  if os.path.isdir(os.path.join(d, 'metrics')))
    if runs:
        return runs[-1]
    if required:
        raise FileNotFoundError(f'No run with metrics under {parent}')
    return None


def req_rows(qps, tag, seed=SEED, name='request_metrics.csv', required=True):
    d = run_dir(qps, tag, seed, required=required)
    if d is None:
        return None
    return list(csv.DictReader(open(os.path.join(d, 'metrics', name))))


def have(qps, tag, seed=SEED):
    return run_dir(qps, tag, seed, required=False) is not None


def pctls(rows, col, scale=1000.0):
    v = np.array([float(r[col]) / scale for r in rows])
    return {'mean': v.mean(), 'p50': np.median(v),
            'p90': np.percentile(v, 90), 'p99': np.percentile(v, 99)}

In [ ]:
# BFCL: median FTR / E2E, baseline vs Sutradhara (DS+KV)
print(f"{'':>5}{'FTR p50':>23}{'E2E p50':>23}")
print(f"{'QPS':>5}" + f"{'BL':>7}{'SD':>7}{'gain':>9}" * 2)
for q in QPS_LIST:
    if not (have(q, TAG_BL) and have(q, TAG_SD)):
        continue
    bl, sd = req_rows(q, TAG_BL), req_rows(q, TAG_SD)
    bf, sf = pctls(bl, 'first_token_render_ms'), pctls(sd, 'first_token_render_ms')
    be, se = pctls(bl, 'e2e_time_ms'), pctls(sd, 'e2e_time_ms')
    line = f"{q:>5}"
    for b, s in ((bf, sf), (be, se)):
        line += f"{b['p50']:>7.2f}{s['p50']:>7.2f}{(b['p50']-s['p50'])/b['p50']*100:>+8.1f}%"
    print(line)